# Lesson 23 Lab — One-shot LLM Pruning: SparseGPT and Wanda Mechanisms

**Puzzle:** Why should calibration activations change which LLM weights survive?

This notebook is designed for a CUDA GPU and retains the output of a complete RTX 5090 run.


## Why this matters

One-shot LLM pruning must choose a support without full retraining. Magnitude ignores input usage; Wanda combines weight magnitude with activation norms; SparseGPT uses second-order reconstruction information and sequential compensation. A small layer can expose these objectives without pretending to reproduce a 70B run.


## 0. Predict before running

1. Predict how rescaling one input feature changes Wanda but not magnitude ranking.
2. Explain what the diagonal-curvature proxy omits from SparseGPT.
3. Choose calibration and held-out splits for a fair one-shot comparison.

For every answer, name the observation that would prove it wrong.


## 1. Name the concrete objects

A wide linear projection, calibration tokens with deliberately uneven feature scales, held-out tokens, magnitude scores, Wanda scores, a diagonal-curvature proxy, and equal-sparsity reconstructed outputs form the lab.

- Calibration activations define feature importance for one-shot pruning.
- Wanda scoring and SparseGPT compensation are not the same algorithm.
- Toy layer reconstruction cannot establish 70B perplexity or speed.


## 2. Derive the mechanism

For `Y=XW^T`, Wanda scores weight `w_ij` by `|w_ij| ||X_:j||`, so a modest weight on a frequently excited feature may outrank a larger unused weight. SparseGPT instead minimizes layer reconstruction with an approximate Hessian and updates remaining weights as columns are pruned. A diagonal `X^T X` proxy can illustrate sensitivity but omits the inverse-Hessian sequential algorithm. Equal sparsity and held-out output error are required for comparison.

Keep value sparsity, physical shape, representation, and runtime evidence separate.


## 3. Verify the execution environment

Inspect the next cell before running it: it asserts CUDA, fixes the seed, defines transparent timing/numerical helpers, and prints the GPU/PyTorch/CUDA record needed to interpret every output.


In [1]:
LESSON_NO = 23
LESSON_TITLE = 'One-shot LLM Pruning: SparseGPT and Wanda Mechanisms'

from pathlib import Path
import copy, gzip, hashlib, importlib.util, io, json, math, random, shutil, statistics, sys
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260808 + LESSON_NO
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name,
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered:
        return float("nan")
    position = (len(ordered) - 1) * q
    lo, hi = math.floor(position), math.ceil(position)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - position) + ordered[hi] * (position - lo)

def cuda_times(fn, warmup=6, repeats=24):
    with torch.inference_mode():
        for _ in range(warmup):
            fn()
        torch.cuda.synchronize()
        samples = []
        for _ in range(repeats):
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            fn()
            end.record()
            end.synchronize()
            samples.append(float(start.elapsed_time(end)))
    return samples

def timing_summary(samples):
    return {
        "median_ms": float(statistics.median(samples)),
        "p95_ms": float(percentile(samples, 0.95)),
        "p99_ms": float(percentile(samples, 0.99)),
        "samples_ms": [float(x) for x in samples],
    }

def count_params(module):
    return int(sum(p.numel() for p in module.parameters()))

def zero_fraction(tensor):
    return float((tensor == 0).float().mean().item())

def magnitude_mask(tensor, sparsity):
    flat = tensor.detach().abs().flatten()
    prune_count = int(round(flat.numel() * float(sparsity)))
    prune_count = min(max(prune_count, 0), flat.numel())
    mask = torch.ones_like(flat)
    if prune_count:
        idx = torch.topk(flat, prune_count, largest=False).indices
        mask[idx] = 0
    return mask.view_as(tensor)

def exact_2_4_mask(weight):
    assert weight.shape[-1] % 4 == 0
    groups = weight.detach().abs().reshape(*weight.shape[:-1], -1, 4)
    keep = torch.topk(groups, 2, dim=-1, largest=True).indices
    mask = torch.zeros_like(groups)
    mask.scatter_(-1, keep, 1)
    return mask.reshape_as(weight)

def compliance_2_4(weight):
    groups = weight.detach().reshape(*weight.shape[:-1], -1, 4)
    return float(((groups != 0).sum(dim=-1) == 2).float().mean().item())

def tensor_metrics(reference, candidate):
    ref = reference.float()
    cand = candidate.float()
    delta = cand - ref
    return {
        "rmse": float(torch.sqrt(torch.mean(delta.square())).item()),
        "mae": float(torch.mean(delta.abs()).item()),
        "max_error": float(delta.abs().max().item()),
        "cosine": float(F.cosine_similarity(ref.flatten(), cand.flatten(), dim=0).item()),
    }

def spearman(a, b):
    a = torch.as_tensor(a, dtype=torch.float64)
    b = torch.as_tensor(b, dtype=torch.float64)
    ra = torch.empty_like(a)
    rb = torch.empty_like(b)
    ra[torch.argsort(a)] = torch.arange(a.numel(), dtype=torch.float64)
    rb[torch.argsort(b)] = torch.arange(b.numel(), dtype=torch.float64)
    ra -= ra.mean(); rb -= rb.mean()
    return float((ra @ rb / (ra.norm() * rb.norm() + 1e-12)).item())


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.12.0",
  "cuda_runtime": "13.0",
  "python": "3.12.13",
  "seed": 20260831
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | plain magnitude one-shot pruning |
| Candidate | Wanda activation-aware scoring and a diagonal-curvature sensitivity proxy |
| Held constant | weights, calibration tokens, held-out tokens, sparsity, grouping policy, and seed |
| Measurements | held-out RMSE, cosine similarity, support overlap, sparsity, and calibration feature scales |
| Evidence | `numerical-model` |

**Experiment:** Compare magnitude, Wanda, and diagonal-curvature masks at identical 50% sparsity on held-out layer outputs.


## 5. Read the experiment code

The notebook constructs calibration features with unequal energy so activation-aware methods have a measurable signal. Each scoring rule retains the same number of weights per row. Output metrics are computed on separate held-out tokens. The artifact explicitly labels the curvature route a proxy rather than SparseGPT.

Do not execute until the code implements the frozen table above.


In [2]:
torch.manual_seed(SEED)
out_features,in_features=512,512
w=torch.randn(out_features,in_features,device=DEVICE); scales=torch.logspace(-1,1,in_features,device=DEVICE); cal=torch.randn(256,in_features,device=DEVICE)*scales; held=torch.randn(256,in_features,device=DEVICE)*scales
def row_mask(score,rate=0.5):
    k=int(score.shape[1]*(1-rate)); idx=torch.topk(score,k,dim=1).indices; m=torch.zeros_like(score); m.scatter_(1,idx,1); return m
mag_score=w.abs(); act_norm=cal.square().mean(0).sqrt(); wanda_score=w.abs()*act_norm
h=(cal.T@cal)/cal.shape[0]+0.01*torch.eye(in_features,device=DEVICE); hinv=torch.linalg.inv(h); curvature_score=w.square()/(torch.diag(hinv).clamp_min(1e-8))[None,:]
mm=row_mask(mag_score); wm=row_mask(wanda_score); cm=row_mask(curvature_score)
with torch.inference_mode(): ref=F.linear(held,w); my=F.linear(held,w*mm); wy=F.linear(held,w*wm); cy=F.linear(held,w*cm)
me=tensor_metrics(ref,my); we=tensor_metrics(ref,wy); ce=tensor_metrics(ref,cy); overlap=float(((mm==1)&(wm==1)).sum().item()/max((mm==1).sum().item(),1))
metrics={"sparsity":zero_fraction(w*mm),"magnitude_rmse":me["rmse"],"wanda_rmse":we["rmse"],"curvature_rmse":ce["rmse"],"magnitude_cosine":me["cosine"],"wanda_cosine":we["cosine"],"curvature_cosine":ce["cosine"],"magnitude_wanda_overlap":overlap,"calibration_scale_min":float(scales.min().item()),"calibration_scale_max":float(scales.max().item())}
analysis=(f"At {metrics['sparsity']:.1%} sparsity, magnitude/Wanda/curvature-proxy held-out RMSE values were "
          f"{me['rmse']:.6f}/{we['rmse']:.6f}/{ce['rmse']:.6f}. Magnitude and Wanda retained-support overlap was "
          f"{overlap:.1%} under a 100x calibration feature-scale range. The curvature score is a diagonal OBS-style proxy, not SparseGPT's sequential algorithm.")


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Magnitude RMSE | 20.097502 |
| Wanda RMSE | 4.029922 |
| Curvature-proxy RMSE | 6.004300 |
| Magnitude cosine | 0.962958 |
| Wanda cosine | 0.998537 |
| Support overlap | 66.47% |


## 7. Interpret rather than merely print

At 50.0% sparsity, magnitude/Wanda/curvature-proxy held-out RMSE values were 20.097502/4.029922/6.004300. Magnitude and Wanda retained-support overlap was 66.5% under a 100x calibration feature-scale range. The curvature score is a diagonal OBS-style proxy, not SparseGPT's sequential algorithm.

The result is bounded to the shapes, seed, packages, and evidence label printed here.


## 8. Keep the evidence label honest

This run is labeled **`numerical-model`**. The CUDA experiment isolates a numerical mechanism. It is not a full paper reproduction, trained production model, or native sparse-kernel benchmark.

The next cell writes the canonical JSON artifact and prints the same payload.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 23,
    "title": 'One-shot LLM Pruning: SparseGPT and Wanda Mechanisms',
    "environment": ENV,
    "evidence_label": 'numerical-model',
    "metrics": metrics,
    "analysis": analysis,
    "conclusion": 'Activation-aware support selection can reduce one-shot reconstruction error, but official algorithms and full-model evidence remain separate gates.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 23,
  "title": "One-shot LLM Pruning: SparseGPT and Wanda Mechanisms",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.12.0",
    "cuda_runtime": "13.0",
    "python": "3.12.13",
    "seed": 20260831
  },
  "evidence_label": "numerical-model",
  "metrics": {
    "sparsity": 0.5,
    "magnitude_rmse": 20.097501754760742,
    "wanda_rmse": 4.029922008514404,
    "curvature_rmse": 6.004299640655518,
    "magnitude_cosine": 0.9629581570625305,
    "wanda_cosine": 0.9985371828079224,
    "curvature_cosine": 0.9967498183250427,
    "magnitude_wanda_overlap": 0.6647262573242188,
    "calibration_scale_min": 0.10000000149011612,
    "calibration_scale_max": 10.0
  },
  "analysis": "At 50.0% sparsity, magnitude/Wanda/curvature-proxy held-out RMSE values were 20.097502/4.029922/6.004300. Magnitude and Wanda retained-support overlap was 66.5% under a 100x calibration feature-scale range. The curvature score is a diagonal OBS

## 9. Make the bounded decision

> Activation-aware support selection can reduce one-shot reconstruction error, but official algorithms and full-model evidence remain separate gates.

**Acceptance/rollback:** Accept an LLM pruning method only after frozen calibration, full-model perplexity/zero-shot gates, serialization, and a supported sparse inference path are measured.

**Failure analysis:** Calibration domains can bias activation norms, and per-row toy masks omit blockwise sequential compensation. Lower layer RMSE may not preserve generation, rare capabilities, or safety. Unstructured zeros may still run dense.


## 10. Extend the evidence

Run official SparseGPT and Wanda implementations on a pinned open model, sweep calibration domains and sparsity patterns, then benchmark a named sparse runtime separately from quality.

The full evidence boundary and references are in [`README.md`](README.md).
